# Level 3 — Neural Intent Classifier

End-to-end notebook: train a lightweight embedding-based intent model with PyTorch, then pipe predictions through the Level 3 neuro-symbolic pipeline (symbol emission → rule engine → decision).

**Architecture**: Utterance → Token Embedding → Mean Pooling → Linear → Softmax → `NeuralIntentModel` → `SymbolEmitter` → `RuleEngine` → Decision

## Cell 1 — Imports

Standard scientific stack plus PyTorch training utilities and the two Level 3 modules: `NeuralIntentModel` (embedding + linear classifier) and `Level3Pipeline` (neural → symbol → rule chain).

In [ ]:
import os
import sys
import importlib

# Ensure project root is on sys.path so level3.* imports resolve
_repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re

# ------------------------------------------------------------------
# Purge any stale/broken level3 module entries from sys.modules so
# Python re-imports every .py file from disk on the next import.
# This is safer than importlib.reload when a prior import failed
# and left a partial module object in sys.modules.
# ------------------------------------------------------------------
for _mod_key in list(sys.modules.keys()):
    if _mod_key == "level3" or _mod_key.startswith("level3."):
        del sys.modules[_mod_key]

# Fresh imports in dependency order
import level3.intent_constants
import level3.symbol_schema
import level3.symbol_emitter
import level3.neural_intent_model
import level3.rule_engine
import level3.level3_pipeline

from level3.neural_intent_model import NeuralIntentModel
from level3.level3_pipeline import Level3Pipeline

print("Imports OK")
print("PyTorch version:", torch.__version__)


ImportError: cannot import name 'RuleEngine' from 'level3.rule_engine' (c:\git\nsai_poc\level3\rule_engine.py)

## Cell 2 — Load Dataset

Reads the shared intent dataset. Expected columns: `utterance` (input text) and `intent` (one of `summarization`, `execution`, `investigate`, `out_of_scope`).

In [ ]:
# Dataset lives one level up in data/
DATA_PATH = os.path.join(_repo_root, "data", "intents_base.csv")

df = pd.read_csv(DATA_PATH)

assert set(df.columns) >= {"utterance", "intent"}, f"Unexpected columns: {df.columns.tolist()}"
df["intent"] = df["intent"].str.lower().str.strip()

print("Shape:", df.shape)
print("Intent distribution:")
print(df["intent"].value_counts())
df.head()

## Cell 3 — Label Mapping

Maps string intent names to integer indices required by `nn.CrossEntropyLoss`. The reverse dict `index_to_intent` is used during inference to convert predicted indices back to readable labels.

In [ ]:
intent_to_index = {
    "summarization": 0,
    "execution": 1,
    "investigate": 2,
    "out_of_scope": 3,
}

index_to_intent = {v: k for k, v in intent_to_index.items()}

df["label"] = df["intent"].map(intent_to_index)

# Drop any rows with intents not in our mapping
unmapped = df["label"].isna().sum()
if unmapped:
    print(f"Warning: {unmapped} rows with unmapped intents dropped")
    df = df.dropna(subset=["label"])

df["label"] = df["label"].astype(int)

print("Label distribution:")
print(df["label"].value_counts().sort_index())
df.head()

## Cell 4 — Simple Tokenizer

Deterministic, reproducible tokenizer: lowercase → strip non-alphanumeric → split on whitespace. No external dependencies.

In [ ]:
def tokenize(text: str):
    text = text.lower()
    text = re.sub(r"[^a-z0-9 ]", "", text)
    return text.split()

# Quick sanity check
print(tokenize("Restart the message-queue consumers in PROD!"))


## Cell 5 — Build Vocabulary

Builds a frequency-ordered word index over the full dataset. Index 0 is reserved for `<PAD>` so embedding lookups on padding tokens produce zero-like vectors.

In [ ]:
all_tokens = []
for utterance in df["utterance"]:
    all_tokens.extend(tokenize(utterance))

counter = Counter(all_tokens)

# Reserve index 0 for <PAD>
vocab = {word: idx + 1 for idx, (word, _) in enumerate(counter.most_common())}
vocab["<PAD>"] = 0

vocab_size = len(vocab)
print("Vocab size:", vocab_size)
print("Top 10 tokens:", counter.most_common(10))

## Cell 6 — Encode Sentences

Converts a raw string into a fixed-length integer sequence of `MAX_LEN=12`. Tokens missing from the vocabulary map to 0 (`<PAD>`). Sequences shorter than `MAX_LEN` are right-padded; longer ones are truncated.

In [ ]:
MAX_LEN = 12

def encode(text: str):
    tokens = tokenize(text)
    ids = [vocab.get(token, 0) for token in tokens]
    if len(ids) < MAX_LEN:
        ids += [0] * (MAX_LEN - len(ids))
    else:
        ids = ids[:MAX_LEN]
    return ids

# Quick sanity check
print(encode("restart the message queue consumers"))

## Cell 7 — Custom Dataset & DataLoader

Wraps the DataFrame in a PyTorch `Dataset` so the training loop can batch and shuffle automatically. Each item is an `(input_ids, label)` pair of tensors.

In [ ]:
class IntentDataset(Dataset):
    def __init__(self, df):
        self.inputs = df["utterance"].tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        input_ids = torch.tensor(encode(self.inputs[idx]), dtype=torch.long)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return input_ids, label


dataset = IntentDataset(df)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

print(f"Dataset size: {len(dataset)} samples")
print(f"Batches per epoch: {len(loader)}")

# Peek at a single batch
sample_ids, sample_labels = next(iter(loader))
print("Sample input_ids shape:", sample_ids.shape)
print("Sample labels:", sample_labels)

## Cell 8 — Initialize Model

`NeuralIntentModel` uses an `nn.Embedding` layer (size `vocab_size × EMBED_DIM`) followed by mean-pooling and a linear output head. Loss: multi-class cross-entropy. Optimizer: Adam with lr=0.01.

In [ ]:
EMBED_DIM = 32
NUM_CLASSES = 4

model = NeuralIntentModel(
    vocab_size=vocab_size,
    embed_dim=EMBED_DIM,
    num_classes=NUM_CLASSES,
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal trainable parameters: {total_params:,}")

## Cell 9 — Training Loop

40 epochs with mini-batch gradient descent. Loss is printed every 10 epochs so you can verify it is decreasing.

In [ ]:
EPOCHS = 40

model.train()
for epoch in range(EPOCHS):
    total_loss = 0.0

    for input_ids, labels in loader:
        optimizer.zero_grad()
        logits = model(input_ids)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1:>3}/{EPOCHS}  Loss: {total_loss:.4f}")

print("\nTraining complete.")

## Cell 10 — Save Model

Persists the trained weights to `level3_intent_model.pt` in the `level3/` folder so the pipeline can reload them without re-training.

In [ ]:
MODEL_PATH = os.path.join(os.path.dirname(os.getcwd()), "level3", "level3_intent_model.pt")
# Save relative to the level3 directory (notebook cwd = level3/)
MODEL_PATH = "level3_intent_model.pt"

torch.save(model.state_dict(), MODEL_PATH)
print(f"Model saved to: {os.path.abspath(MODEL_PATH)}")

## Cell 11 — Run Through Level3Pipeline

`Level3Pipeline` chains the trained `NeuralIntentModel` → `SymbolEmitter` → `RuleEngine`. Loading from the saved state dict puts the model in eval mode automatically.

In [ ]:
pipeline = Level3Pipeline(
    vocab_size=vocab_size,
    embed_dim=EMBED_DIM,
    num_classes=NUM_CLASSES,
)

pipeline.load_model_state(MODEL_PATH)
print("Pipeline loaded and model set to eval mode.")

## Cell 12 — Inference Helper

Encodes a raw string into a `(1, MAX_LEN)` tensor and passes it through the full pipeline, returning `{symbol, decision}` where `decision` contains the rule-engine action and `requires_approval` flag.

In [ ]:
def run_inference(text: str) -> dict:
    input_ids = torch.tensor([encode(text)], dtype=torch.long)  # shape: (1, MAX_LEN)
    result = pipeline.process(input_ids)
    return result

# Quick smoke test
print(run_inference("restart nginx in prod"))

## Cell 13 — Test Examples & Validation

Runs the four canonical test utterances through the pipeline. Each result should produce a distinct symbol and rule-engine action. Assertions at the end verify the expected symbols, confirming end-to-end correctness.

In [ ]:
examples = [
    ("restart the message queue consumers",                    "execution"),
    ("summarize the scheduled downtime impact on availability sla", "summarization"),
    ("why is the backup not completing within the window",     "investigate"),
    ("tell me about travel",                                   "out_of_scope"),
]

VALID_SYMBOLS = set(intent_to_index.keys())
all_passed = True

for text, expected_symbol in examples:
    result = run_inference(text)
    symbol   = result["symbol"]
    decision = result["decision"]
    status   = "✅" if symbol == expected_symbol else "⚠️ "
    if symbol != expected_symbol:
        all_passed = False
    print(f"{status} Input   : {text}")
    print(f"   Expected: {expected_symbol}")
    print(f"   Got     : {symbol}")
    print(f"   Action  : {decision['action']}  |  requires_approval={decision['requires_approval']}")
    print("-" * 60)

print()
if all_passed:
    print("✅  All examples matched expected symbols — pipeline OK")
else:
    print("⚠️  Some examples did not match — check training convergence or label mapping")